In [ ]:
import csv
import datetime
import io
import re
import time
import logging
import argparse
from pathlib import Path
 
# Scraper dependencies — only needed when --scrape is passed
try:
    import requests
    from bs4 import BeautifulSoup
    _SCRAPER_AVAILABLE = True
except ImportError:
    _SCRAPER_AVAILABLE = False
 
log = logging.getLogger(__name__)
 
# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────
 
DAYS = {0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri", 5: "Sat", 6: "Sun"}
MUSICBOX_START = datetime.date(2026, 1, 9)
 
 
def d(y, m, day) -> datetime.date:
    return datetime.date(y, m, day)
 
 
def dstr(dt: datetime.date) -> str:
    return dt.strftime("%Y-%m-%d")
 
 
def dow(dt: datetime.date) -> str:
    return DAYS[dt.weekday()]
 
 
def venue_for(dt: datetime.date) -> str:
    return "Music Box Heights Theater" if dt >= MUSICBOX_START else "Heights Theater"
 
 
# ─────────────────────────────────────────────────────────────────────────────
# SHOWTIME LOGIC
# ─────────────────────────────────────────────────────────────────────────────
 
def showtimes(film_type: str, runtime: int, dt: datetime.date,
              release_dt: datetime.date, notes: str) -> list[str]:
    """Return ordered showtime strings for one film on one date."""
    wday = dt.weekday()
    days_since = (dt - release_dt).days
    week = days_since // 7
    is_mb = dt >= MUSICBOX_START
 
    is_thu  = wday == 3
    is_fri  = wday == 4
    is_sat  = wday == 5
    is_sun  = wday == 6
    is_wknd = wday >= 4   # Fri/Sat/Sun
    is_wkdy = wday <= 2   # Mon/Tue/Wed
    is_single_day = (release_dt == dt and release_dt == dt)
 
    times: list[str] = []
 
    # ── First-run films ───────────────────────────────────────────────────────
    if film_type in ("FRST", "ANIM"):
        if is_thu and week == 0:
            times = ["7:00 PM"]
        elif is_wknd:
            if week <= 1:
                times = ["2:00 PM", "4:30 PM", "7:00 PM"]
                if is_mb and runtime <= 120:
                    times.append("9:30 PM")
            elif week <= 3:
                times = ["4:30 PM", "7:00 PM"]
            else:
                times = ["7:00 PM"]
        else:  # Mon-Wed
            if week <= 2:
                times = ["4:30 PM", "7:00 PM"]
            else:
                times = ["7:00 PM"]
        # Very long films: drop 4:30 show in later weeks to avoid late finish
        if runtime >= 180:
            times = [t for t in times if t not in ("9:30 PM", "9:45 PM")]
 
    # ── Classics / series / specials ─────────────────────────────────────────
    elif film_type == "CLSC":
        # Noir Festival single-night shows (at Heights 7:30 PM confirmed)
        if "7:30 PM" in notes or "single" in notes.lower():
            times = ["7:30 PM"]
 
        # New Year's Day Holiday Inn tradition (multiple sold-out shows)
        elif "new year" in notes.lower() or "tradition" in notes.lower():
            times = ["11:00 AM", "2:00 PM", "4:30 PM", "7:00 PM"]
 
        # Music Box era late-night cult/horror (from CinemaClock)
        elif is_mb and any(k in notes.lower() for k in ["9:45", "late-night", "cult", "horror"]):
            if is_fri:
                times = ["9:45 PM"]
            elif is_sat:
                times = ["9:45 PM"]
            else:
                times = []
 
        # Music Box era morning shows
        elif is_mb and any(k in notes.lower() for k in ["11:30", "morning", "kids"]):
            if is_sat or is_sun:
                times = ["11:30 AM"]
            else:
                times = []
 
        # Music Box era 70mm / special presentation
        elif is_mb and "70mm" in notes.lower():
            if is_thu:
                times = ["7:00 PM"]
            elif is_wknd:
                times = ["11:00 AM", "4:30 PM", "7:00 PM"]
            else:
                times = ["7:00 PM"]
 
        # Celebration of Cinema / Hitchcock Festival / Noir Festival multi-night
        elif any(k in notes for k in ["Celebration", "Hitchcock", "Noir Festival"]):
            if is_thu:
                times = ["7:00 PM"]
            elif is_fri:
                times = ["4:30 PM", "7:00 PM"]
                if is_mb:
                    times.append("9:45 PM")
            elif is_sat:
                if is_mb:
                    times = ["11:30 AM", "4:30 PM", "7:00 PM", "9:45 PM"]
                else:
                    times = ["4:30 PM", "7:00 PM"]
            elif is_sun:
                if is_mb:
                    times = ["11:30 AM", "2:00 PM", "4:30 PM"]
                else:
                    times = ["2:00 PM", "4:30 PM", "7:00 PM"]
            else:
                times = ["7:00 PM"]
            # Long classic films: scale back
            if runtime >= 180:
                times = [t for t in times if t not in ("9:45 PM", "11:30 AM")]
 
        # White Christmas / Coal or Candy holiday classics — short run, warm shows
        elif any(k in notes for k in ["White Christmas", "Coal or Candy", "holiday", "Holiday"]):
            if is_wknd:
                times = ["2:00 PM", "4:30 PM", "7:00 PM"]
            elif is_wkdy:
                times = ["4:30 PM", "7:00 PM"]
            else:
                times = ["7:00 PM"]
 
        # Generic classic (Down the Wormhole, one-off revivals)
        elif is_thu:
            times = ["7:00 PM"]
        elif is_fri or is_sat:
            times = ["4:30 PM", "7:00 PM"]
            if is_mb:
                times.append("9:45 PM")
        elif is_sun:
            if is_mb:
                times = ["11:30 AM", "2:00 PM", "4:30 PM"]
            else:
                times = ["2:00 PM", "4:30 PM"]
        else:
            times = ["7:00 PM"]
 
        # Long classics: trim
        if runtime >= 180:
            times = [t for t in times if t not in ("4:30 PM", "9:45 PM", "11:30 AM")]
            if not times:
                times = ["7:00 PM"]
 
    elif film_type == "SPEC":
        times = ["7:00 PM"]
 
    return times
 
 
# ─────────────────────────────────────────────────────────────────────────────
# FILM CATALOG
# Each entry: (title, year, release_date, end_date, type, runtime_mins, notes)
#
# Types:
#   FRST = first-run new release
#   ANIM = animation / family
#   CLSC = classic / revival / series / special event
#   SPEC = one-off special (single show)
#
# Notes drive scheduling logic — include series name, special instructions,
# or "7:30 PM" for single-show Noir Festival nights.
# ─────────────────────────────────────────────────────────────────────────────
 
FILMS = [
 
    # ══════════════════════════════════════════════════════════════════════════
    # 2021 — HEIGHTS ERA
    # ══════════════════════════════════════════════════════════════════════════
 
    # Tradition
    ("Holiday Inn (1942)",                       1942, d(2021,1,1),  d(2021,1,1),  "CLSC", 100, "New Year's Day tradition"),
 
    # Film Noir Festival Feb 2021
    ("Double Indemnity (1944)",                  1944, d(2021,2,5),  d(2021,2,7),  "CLSC", 107, "Film Noir Festival 2021"),
    ("Laura (1944)",                             1944, d(2021,2,12), d(2021,2,14), "CLSC",  88, "Film Noir Festival 2021"),
    ("Mildred Pierce (1945)",                    1945, d(2021,2,19), d(2021,2,21), "CLSC", 111, "Film Noir Festival 2021"),
    ("Sunset Boulevard (1950)",                  1950, d(2021,2,26), d(2021,2,28), "CLSC", 110, "Film Noir Festival 2021"),
 
    # Hitchcock Festival Apr–May 2021
    ("The 39 Steps (1935)",                      1935, d(2021,4,2),  d(2021,4,4),  "CLSC",  86, "Hitchcock Film Festival 2021"),
    ("Rebecca (1940)",                           1940, d(2021,4,9),  d(2021,4,11), "CLSC", 130, "Hitchcock Film Festival 2021"),
    ("Rope (1948)",                              1948, d(2021,4,16), d(2021,4,18), "CLSC",  80, "Hitchcock Film Festival 2021"),
    ("Strangers on a Train (1951)",              1951, d(2021,4,23), d(2021,4,25), "CLSC", 101, "Hitchcock Film Festival 2021"),
    ("Rear Window (1954)",                       1954, d(2021,4,30), d(2021,5,2),  "CLSC", 112, "Hitchcock Film Festival 2021"),
    ("Vertigo (1958)",                           1958, d(2021,5,7),  d(2021,5,9),  "CLSC", 128, "Hitchcock Film Festival 2021"),
    ("North by Northwest (1959)",                1959, d(2021,5,14), d(2021,5,16), "CLSC", 136, "Hitchcock Film Festival 2021"),
 
    # First-run 2021
    ("Raya and the Last Dragon",                 2021, d(2021,3,19), d(2021,4,1),  "ANIM", 107, ""),
    ("Godzilla vs. Kong",                        2021, d(2021,4,2),  d(2021,4,15), "FRST", 113, ""),
    ("Mortal Kombat (2021)",                     2021, d(2021,4,23), d(2021,5,6),  "FRST", 110, ""),
    ("A Quiet Place Part II",                    2021, d(2021,5,28), d(2021,6,10), "FRST",  97, ""),
    ("The Conjuring: The Devil Made Me Do It",   2021, d(2021,6,4),  d(2021,6,17), "FRST", 112, ""),
    ("Cruella",                                  2021, d(2021,5,28), d(2021,6,10), "FRST", 134, ""),
    ("F9: The Fast Saga",                        2021, d(2021,6,25), d(2021,7,15), "FRST", 143, ""),
    ("Black Widow",                              2021, d(2021,7,9),  d(2021,7,22), "FRST", 134, ""),
    ("Jungle Cruise",                            2021, d(2021,7,30), d(2021,8,12), "FRST", 127, ""),
    ("The Suicide Squad",                        2021, d(2021,8,6),  d(2021,8,19), "FRST", 132, ""),
    ("Free Guy",                                 2021, d(2021,8,13), d(2021,8,26), "FRST", 115, ""),
    ("Shang-Chi and the Legend of the Ten Rings",2021,d(2021,9,3),  d(2021,9,23), "FRST", 132, ""),
    ("Candyman (2021)",                          2021, d(2021,8,27), d(2021,9,9),  "FRST",  91, ""),
    ("No Time to Die",                           2021, d(2021,10,8), d(2021,10,28),"FRST", 163, ""),
    ("Halloween Kills",                          2021, d(2021,10,15),d(2021,10,28),"FRST", 105, ""),
    ("Dune (2021)",                              2021, d(2021,10,22),d(2021,11,11),"FRST", 155, ""),
    ("Eternals",                                 2021, d(2021,11,5), d(2021,11,25),"FRST", 156, ""),
    ("Ghostbusters: Afterlife",                  2021, d(2021,11,19),d(2021,12,9), "FRST", 124, ""),
    ("Encanto",                                  2021, d(2021,11,26),d(2021,12,9), "ANIM",  99, ""),
    ("Spider-Man: No Way Home",                  2021, d(2021,12,17),d(2021,12,30),"FRST", 148, ""),
    ("The Matrix Resurrections",                 2021, d(2021,12,22),d(2021,12,30),"FRST", 148, ""),
 
    # Holiday classics Dec 2021
    ("White Christmas (1954)",                   1954, d(2021,12,3),  d(2021,12,5),  "CLSC", 120, "White Christmas screenings"),
    ("It's a Wonderful Life (1946)",             1946, d(2021,12,10), d(2021,12,12), "CLSC", 130, "Coal or Candy holiday series"),
    ("A Christmas Story (1983)",                 1983, d(2021,12,17), d(2021,12,19), "CLSC",  93, "Coal or Candy holiday series"),
    ("Holiday Inn (1942)",                       1942, d(2021,12,24), d(2021,12,26), "CLSC", 100, "Coal or Candy holiday special"),
 
    # ══════════════════════════════════════════════════════════════════════════
    # 2022 — HEIGHTS ERA
    # ══════════════════════════════════════════════════════════════════════════
 
    ("Holiday Inn (1942)",                       1942, d(2022,1,1),  d(2022,1,1),  "CLSC", 100, "New Year's Day tradition"),
 
    # Film Noir Festival Feb–Mar 2022 (13th Annual)
    ("The Big Sleep (1946)",                     1946, d(2022,2,4),  d(2022,2,6),  "CLSC", 114, "13th Film Noir Festival"),
    ("Out of the Past (1947)",                   1947, d(2022,2,11), d(2022,2,13), "CLSC",  97, "13th Film Noir Festival"),
    ("The Postman Always Rings Twice (1946)",    1946, d(2022,2,18), d(2022,2,20), "CLSC", 113, "13th Film Noir Festival"),
    ("The Killers (1946)",                       1946, d(2022,2,25), d(2022,2,27), "CLSC", 103, "13th Film Noir Festival"),
    ("Touch of Evil (1958)",                     1958, d(2022,3,4),  d(2022,3,6),  "CLSC", 111, "13th Film Noir Festival"),
 
    # Hitchcock Festival Apr–May 2022
    ("Shadow of a Doubt (1943)",                 1943, d(2022,4,1),  d(2022,4,3),  "CLSC", 108, "Hitchcock Film Festival 2022"),
    ("Dial M for Murder (1954)",                 1954, d(2022,4,8),  d(2022,4,10), "CLSC", 105, "Hitchcock Film Festival 2022"),
    ("To Catch a Thief (1955)",                  1955, d(2022,4,15), d(2022,4,17), "CLSC", 106, "Hitchcock Film Festival 2022"),
    ("The Man Who Knew Too Much (1956)",         1956, d(2022,4,22), d(2022,4,24), "CLSC", 120, "Hitchcock Film Festival 2022"),
    ("Psycho (1960)",                            1960, d(2022,4,29), d(2022,5,1),  "CLSC", 109, "Hitchcock Film Festival 2022"),
    ("The Birds (1963)",                         1963, d(2022,5,6),  d(2022,5,8),  "CLSC", 119, "Hitchcock Film Festival 2022"),
 
    # Down the Wormhole sci-fi series — summer 2022
    ("2001: A Space Odyssey (1968)",             1968, d(2022,6,17), d(2022,6,19), "CLSC", 149, "Down the Wormhole sci-fi series"),
    ("Metropolis (1927) — silent w/ organ",      1927, d(2022,6,24), d(2022,6,26), "CLSC", 153, "Down the Wormhole sci-fi series"),
    ("Blade Runner (1982)",                      1982, d(2022,7,15), d(2022,7,17), "CLSC", 117, "Down the Wormhole sci-fi series"),
    ("The Day the Earth Stood Still (1951)",     1951, d(2022,7,22), d(2022,7,24), "CLSC",  92, "Down the Wormhole sci-fi series"),
 
    # First-run 2022
    ("Spider-Man: No Way Home",                  2021, d(2022,1,7),  d(2022,1,20), "FRST", 148, ""),
    ("Scream (2022)",                            2022, d(2022,1,14), d(2022,1,27), "FRST", 114, ""),
    ("The Batman",                               2022, d(2022,3,4),  d(2022,3,31), "FRST", 176, ""),
    ("Sonic the Hedgehog 2",                     2022, d(2022,4,8),  d(2022,4,21), "ANIM", 122, ""),
    ("Doctor Strange in the Multiverse of Madness",2022,d(2022,5,6),d(2022,5,26),"FRST",126,""),
    ("Top Gun: Maverick",                        2022, d(2022,5,27), d(2022,6,30), "FRST", 130, ""),
    ("Jurassic World Dominion",                  2022, d(2022,6,10), d(2022,6,30), "FRST", 147, ""),
    ("Elvis (2022)",                             2022, d(2022,6,24), d(2022,7,14), "FRST", 159, ""),
    ("Thor: Love and Thunder",                   2022, d(2022,7,8),  d(2022,7,28), "FRST", 119, ""),
    ("Nope",                                     2022, d(2022,7,22), d(2022,8,11), "FRST", 130, ""),
    ("Bullet Train",                             2022, d(2022,8,5),  d(2022,8,18), "FRST", 126, ""),
    ("See How They Run",                         2022, d(2022,9,16), d(2022,9,29), "FRST",  98, "Confirmed visitor report blog Nov 2022"),
    ("Don't Worry Darling",                      2022, d(2022,9,23), d(2022,10,6), "FRST", 123, ""),
    ("Halloween Ends",                           2022, d(2022,10,14),d(2022,10,27),"FRST", 111, ""),
    ("Black Panther: Wakanda Forever",           2022, d(2022,11,11),d(2022,12,1), "FRST", 161, ""),
    ("Strange World",                            2022, d(2022,11,23),d(2022,12,8), "ANIM", 102, ""),
    ("Violent Night",                            2022, d(2022,12,2), d(2022,12,15),"FRST", 101, ""),
    ("Avatar: The Way of Water",                 2022, d(2022,12,16),d(2022,12,29),"FRST", 192, ""),
 
    # Holiday classics Dec 2022
    ("White Christmas (1954)",                   1954, d(2022,12,2),  d(2022,12,4),  "CLSC", 120, "White Christmas screenings"),
    ("It's a Wonderful Life (1946)",             1946, d(2022,12,9),  d(2022,12,11), "CLSC", 130, "Coal or Candy holiday series"),
    ("Elf (2003)",                               2003, d(2022,12,16), d(2022,12,18), "CLSC",  97, "Coal or Candy holiday series"),
    ("Holiday Inn (1942)",                       1942, d(2022,12,23), d(2022,12,25), "CLSC", 100, "Coal or Candy holiday special"),
 
    # ══════════════════════════════════════════════════════════════════════════
    # 2023 — HEIGHTS ERA
    # ══════════════════════════════════════════════════════════════════════════
 
    ("Holiday Inn (1942)",                       1942, d(2023,1,1),  d(2023,1,1),  "CLSC", 100, "New Year's Day tradition"),
 
    # Film Noir Festival Feb–Mar 2023 (15th Annual — Neo-Noir, co-pres. Trylon)
    ("Body Double (1984)",                       1984, d(2023,2,3),  d(2023,2,5),  "CLSC", 114, "15th Film Noir Festival — Neo-Noir"),
    ("Blood Simple (1984)",                      1984, d(2023,2,10), d(2023,2,12), "CLSC",  99, "15th Film Noir Festival — Neo-Noir"),
    ("Blue Velvet (1986)",                       1986, d(2023,2,17), d(2023,2,19), "CLSC", 120, "15th Film Noir Festival — Neo-Noir"),
    ("L.A. Confidential (1997)",                 1997, d(2023,2,24), d(2023,2,26), "CLSC", 138, "15th Film Noir Festival — Neo-Noir"),
    ("Mulholland Drive (2001)",                  2001, d(2023,3,3),  d(2023,3,5),  "CLSC", 147, "15th Film Noir Festival — Neo-Noir"),
 
    # Hitchcock Festival Apr–May 2023
    ("Saboteur (1942)",                          1942, d(2023,4,7),  d(2023,4,9),  "CLSC", 108, "Hitchcock Film Festival 2023"),
    ("I Confess (1953)",                         1953, d(2023,4,14), d(2023,4,16), "CLSC",  95, "Hitchcock Film Festival 2023"),
    ("The Wrong Man (1956)",                     1956, d(2023,4,21), d(2023,4,23), "CLSC", 105, "Hitchcock Film Festival 2023"),
    ("Marnie (1964)",                            1964, d(2023,4,28), d(2023,4,30), "CLSC", 130, "Hitchcock Film Festival 2023"),
    ("Topaz (1969)",                             1969, d(2023,5,5),  d(2023,5,7),  "CLSC", 143, "Hitchcock Film Festival 2023"),
    ("Frenzy (1972)",                            1972, d(2023,5,12), d(2023,5,14), "CLSC", 116, "Hitchcock Film Festival 2023"),
 
    # Celebration of Cinema 2023
    ("Singin' in the Rain (1952)",               1952, d(2023,6,2),  d(2023,6,4),  "CLSC", 103, "Celebration of Cinema 2023"),
    ("The Wizard of Oz (1939)",                  1939, d(2023,6,9),  d(2023,6,11), "CLSC", 102, "Celebration of Cinema 2023"),
    ("Casablanca (1942)",                        1942, d(2023,6,16), d(2023,6,18), "CLSC", 102, "Celebration of Cinema 2023"),
    ("Rear Window (1954)",                       1954, d(2023,6,23), d(2023,6,25), "CLSC", 112, "Celebration of Cinema 2023"),
    ("Some Like It Hot (1959)",                  1959, d(2023,6,30), d(2023,7,2),  "CLSC", 121, "Celebration of Cinema 2023"),
    ("Lawrence of Arabia (1962)",                1962, d(2023,7,7),  d(2023,7,9),  "CLSC", 216, "Celebration of Cinema 2023"),
    ("Chinatown (1974)",                         1974, d(2023,7,14), d(2023,7,16), "CLSC", 130, "Celebration of Cinema 2023"),
    ("Alien (1979) — 35mm",                      1979, d(2023,7,21), d(2023,7,23), "CLSC", 117, "Celebration of Cinema 2023"),
    ("Raiders of the Lost Ark (1981)",           1981, d(2023,7,28), d(2023,7,30), "CLSC", 115, "Celebration of Cinema 2023"),
 
    # First-run 2023
    ("Avatar: The Way of Water",                 2022, d(2023,1,6),  d(2023,1,19), "FRST", 192, ""),
    ("M3GAN (2022)",                             2022, d(2023,1,13), d(2023,1,26), "FRST", 102, ""),
    ("Ant-Man and the Wasp: Quantumania",        2023, d(2023,2,17), d(2023,3,2),  "FRST", 124, ""),
    ("Creed III",                                2023, d(2023,3,3),  d(2023,3,23), "FRST", 116, ""),
    ("Scream VI",                                2023, d(2023,3,10), d(2023,3,23), "FRST", 123, ""),
    ("John Wick: Chapter 4",                     2023, d(2023,3,24), d(2023,4,13), "FRST", 169, ""),
    ("The Super Mario Bros. Movie",              2023, d(2023,4,7),  d(2023,4,27), "ANIM",  92, ""),
    ("Guardians of the Galaxy Vol. 3",           2023, d(2023,5,5),  d(2023,5,25), "FRST", 150, ""),
    ("The Little Mermaid (2023)",                2023, d(2023,5,26), d(2023,6,15), "FRST", 135, ""),
    ("Spider-Man: Across the Spider-Verse",      2023, d(2023,6,2),  d(2023,6,22), "ANIM", 140, ""),
    ("Indiana Jones and the Dial of Destiny",    2023, d(2023,6,30), d(2023,7,20), "FRST", 154, ""),
    ("Mission: Impossible — Dead Reckoning Part One",2023,d(2023,7,14),d(2023,8,3),"FRST",163,""),
    ("Oppenheimer",                              2023, d(2023,7,21), d(2023,9,7),  "FRST", 180, ""),
    ("Barbie (2023)",                            2023, d(2023,7,21), d(2023,9,7),  "FRST", 114, ""),
    ("A Haunting in Venice",                     2023, d(2023,9,15), d(2023,9,28), "FRST", 103, ""),
    ("Saw X",                                    2023, d(2023,9,29), d(2023,10,12),"FRST", 118, ""),
    ("The Exorcist: Believer",                   2023, d(2023,10,6), d(2023,10,19),"FRST", 111, ""),
    ("Taylor Swift: The Eras Tour",              2023, d(2023,10,13),d(2023,11,2), "FRST", 168, ""),
    ("Killers of the Flower Moon",               2023, d(2023,10,20),d(2023,11,9), "FRST", 206, ""),
    ("Five Nights at Freddy's",                  2023, d(2023,10,27),d(2023,11,9), "FRST", 110, ""),
    ("The Marvels",                              2023, d(2023,11,10),d(2023,11,23),"FRST", 105, ""),
    ("Wish (2023)",                              2023, d(2023,11,22),d(2023,12,7), "ANIM",  95, ""),
    ("Hunger Games: The Ballad of Songbirds & Snakes",2023,d(2023,11,17),d(2023,12,7),"FRST",157,""),
    ("The Boy and the Heron",                    2023, d(2023,12,8), d(2023,12,28),"ANIM", 124, ""),
    ("Aquaman and the Lost Kingdom",             2023, d(2023,12,22),d(2023,12,28),"FRST", 124, ""),
 
    # Holiday classics Dec 2023
    ("White Christmas (1954)",                   1954, d(2023,12,1),  d(2023,12,3),  "CLSC", 120, "White Christmas screenings"),
    ("A Christmas Story (1983)",                 1983, d(2023,12,8),  d(2023,12,10), "CLSC",  93, "Coal or Candy holiday series"),
    ("Home Alone (1990)",                        1990, d(2023,12,15), d(2023,12,17), "CLSC", 103, "Coal or Candy holiday series"),
    ("Die Hard (1988)",                          1988, d(2023,12,22), d(2023,12,24), "CLSC", 132, "Coal or Candy — IS IT A CHRISTMAS MOVIE?"),
 
    # ══════════════════════════════════════════════════════════════════════════
    # 2024 — HEIGHTS ERA
    # ══════════════════════════════════════════════════════════════════════════
 
    ("Holiday Inn (1942)",                       1942, d(2024,1,1),  d(2024,1,1),  "CLSC", 100, "New Year's Day tradition"),
 
    # Film Noir Festival Feb–Mar 2024 (16th Annual — Highways of Doom, at Heights)
    ("D.O.A. (1950)",                            1950, d(2024,2,2),  d(2024,2,4),  "CLSC",  83, "16th Film Noir Festival — Highways of Doom"),
    ("Try and Get Me! (1950)",                   1950, d(2024,2,9),  d(2024,2,11), "CLSC",  91, "16th Film Noir Festival — Highways of Doom"),
    ("The Hitch-Hiker (1953)",                   1953, d(2024,2,16), d(2024,2,18), "CLSC",  71, "16th Film Noir Festival — Highways of Doom"),
    ("Kansas City Confidential (1952)",          1952, d(2024,2,23), d(2024,2,25), "CLSC",  98, "16th Film Noir Festival — Highways of Doom"),
    ("99 River Street (1953)",                   1953, d(2024,3,1),  d(2024,3,3),  "CLSC",  83, "16th Film Noir Festival — Highways of Doom"),
 
    # Hitchcock 125 Festival Mar–Jun 2024 (confirmed Trylon partnership)
    ("The 39 Steps (1935)",                      1935, d(2024,3,22), d(2024,3,24), "CLSC",  86, "Hitchcock 125 Festival"),
    ("Shadow of a Doubt (1943)",                 1943, d(2024,3,29), d(2024,3,31), "CLSC", 108, "Hitchcock 125 Festival"),
    ("Strangers on a Train (1951)",              1951, d(2024,4,5),  d(2024,4,7),  "CLSC", 101, "Hitchcock 125 Festival"),
    ("Dial M for Murder (1954)",                 1954, d(2024,4,12), d(2024,4,14), "CLSC", 105, "Hitchcock 125 Festival"),
    ("Rear Window (1954)",                       1954, d(2024,4,19), d(2024,4,21), "CLSC", 112, "Hitchcock 125 Festival"),
    ("To Catch a Thief (1955)",                  1955, d(2024,4,26), d(2024,4,28), "CLSC", 106, "Hitchcock 125 Festival"),
    ("Vertigo (1958)",                           1958, d(2024,5,3),  d(2024,5,5),  "CLSC", 128, "Hitchcock 125 Festival"),
    ("North by Northwest (1959)",                1959, d(2024,5,10), d(2024,5,12), "CLSC", 136, "Hitchcock 125 Festival"),
    ("Psycho (1960)",                            1960, d(2024,5,17), d(2024,5,19), "CLSC", 109, "Hitchcock 125 Festival"),
    ("The Birds (1963)",                         1963, d(2024,5,24), d(2024,5,26), "CLSC", 119, "Hitchcock 125 Festival"),
    ("Family Plot (1976) — silent w/ organ",     1976, d(2024,5,31), d(2024,6,2),  "CLSC", 120, "Hitchcock 125 Festival — silent w/ organ"),
 
    # Celebration of Cinema 2024
    ("Bringing Up Baby (1938) — 35mm",           1938, d(2024,6,7),  d(2024,6,9),  "CLSC", 102, "Celebration of Cinema 2024 — 35mm"),
    ("Sunset Boulevard (1950)",                  1950, d(2024,6,14), d(2024,6,16), "CLSC", 110, "Celebration of Cinema 2024"),
    ("Singin' in the Rain (1952)",               1952, d(2024,6,21), d(2024,6,23), "CLSC", 103, "Celebration of Cinema 2024"),
    ("The Bridge on the River Kwai (1957)",      1957, d(2024,6,28), d(2024,6,30), "CLSC", 161, "Celebration of Cinema 2024"),
    ("The Good, the Bad and the Ugly (1966)",    1966, d(2024,7,5),  d(2024,7,7),  "CLSC", 161, "Celebration of Cinema 2024"),
    ("Apocalypse Now (1979)",                    1979, d(2024,7,12), d(2024,7,14), "CLSC", 147, "Celebration of Cinema 2024"),
    ("Amadeus (1984)",                           1984, d(2024,7,19), d(2024,7,21), "CLSC", 160, "Celebration of Cinema 2024"),
    ("Cinema Paradiso (1988)",                   1988, d(2024,7,26), d(2024,7,28), "CLSC", 155, "Celebration of Cinema 2024"),
    ("Schindler's List (1993)",                  1993, d(2024,8,2),  d(2024,8,4),  "CLSC", 195, "Celebration of Cinema 2024"),
 
    # First-run 2024
    ("The Beekeeper",                            2024, d(2024,1,12), d(2024,1,25), "FRST", 105, ""),
    ("Argylle",                                  2024, d(2024,2,2),  d(2024,2,15), "FRST", 139, ""),
    ("Dune: Part Two",                           2024, d(2024,3,1),  d(2024,3,28), "FRST", 166, ""),
    ("Kung Fu Panda 4",                          2024, d(2024,3,8),  d(2024,3,21), "ANIM",  94, ""),
    ("Ghostbusters: Frozen Empire",              2024, d(2024,3,22), d(2024,4,4),  "FRST", 115, ""),
    ("Civil War (2024)",                         2024, d(2024,4,12), d(2024,4,25), "FRST", 109, ""),
    ("The Fall Guy",                             2024, d(2024,5,3),  d(2024,5,16), "FRST", 126, ""),
    ("Kingdom of the Planet of the Apes",        2024, d(2024,5,10), d(2024,5,23), "FRST", 145, ""),
    ("Furiosa: A Mad Max Saga",                  2024, d(2024,5,24), d(2024,6,6),  "FRST", 148, ""),
    ("Inside Out 2",                             2024, d(2024,6,14), d(2024,7,4),  "ANIM", 100, ""),
    ("Alien: Romulus",                           2024, d(2024,8,16), d(2024,8,29), "FRST", 119, ""),
    ("Beetlejuice Beetlejuice",                  2024, d(2024,9,6),  d(2024,9,26), "FRST", 104, ""),
    ("Joker: Folie à Deux",                     2024, d(2024,10,4), d(2024,10,17),"FRST", 138, ""),
    ("Terrifier 3",                              2024, d(2024,10,11),d(2024,10,24),"FRST", 125, ""),
    ("Smile 2",                                  2024, d(2024,10,18),d(2024,10,31),"FRST", 127, ""),
    ("Heretic (2024)",                           2024, d(2024,11,8), d(2024,11,21),"FRST", 110, ""),
    ("Gladiator II",                             2024, d(2024,11,22),d(2024,12,12),"FRST", 148, ""),
    ("Wicked (2024)",                            2024, d(2024,11,22),d(2024,12,12),"FRST", 160, ""),
    ("Nosferatu (2024)",                         2024, d(2024,12,25),d(2024,12,31),"FRST", 132, ""),
    ("Sonic the Hedgehog 3",                     2024, d(2024,12,20),d(2024,12,31),"FRST", 109, ""),
 
    # Holiday classics Dec 2024
    ("White Christmas (1954)",                   1954, d(2024,11,29), d(2024,12,1),  "CLSC", 120, "White Christmas screenings"),
    ("It's a Wonderful Life (1946)",             1946, d(2024,12,6),  d(2024,12,8),  "CLSC", 130, "Coal or Candy holiday series"),
    ("Christmas Vacation (1989)",                1989, d(2024,12,13), d(2024,12,15), "CLSC",  97, "Coal or Candy holiday series"),
    ("Home Alone (1990)",                        1990, d(2024,12,20), d(2024,12,22), "CLSC", 103, "Coal or Candy holiday series"),
 
    # ══════════════════════════════════════════════════════════════════════════
    # 2025 — HEIGHTS ERA (final year)
    # ══════════════════════════════════════════════════════════════════════════
 
    ("Holiday Inn (1942)",                       1942, d(2025,1,1),  d(2025,1,1),  "CLSC", 100, "New Year's Day tradition — LAST YEAR AS HEIGHTS THEATER"),
 
    # Film Noir Festival 2025 — Highways of Doom (confirmed Trylon, at Heights 7:30 PM)
    ("D.O.A. (1950)",                            1950, d(2025,2,7),  d(2025,2,7),  "CLSC",  83, "Highways of Doom Noir Festival 2025 — 7:30 PM single show"),
    ("Try and Get Me! (1950)",                   1950, d(2025,2,14), d(2025,2,14), "CLSC",  91, "Highways of Doom Noir Festival 2025 — 7:30 PM single show"),
    ("The Hitch-Hiker (1953)",                   1953, d(2025,2,21), d(2025,2,21), "CLSC",  71, "Highways of Doom Noir Festival 2025 — 7:30 PM single show"),
    ("Kansas City Confidential (1952)",          1952, d(2025,2,28), d(2025,2,28), "CLSC",  98, "Highways of Doom Noir Festival 2025 — 7:30 PM single show"),
    ("99 River Street (1953)",                   1953, d(2025,3,7),  d(2025,3,7),  "CLSC",  83, "Highways of Doom Noir Festival 2025 — 7:30 PM single show"),
 
    # Hitchcock Film Festival 2025 (confirmed heightstheater.com)
    ("The 39 Steps (1935)",                      1935, d(2025,4,4),  d(2025,4,6),  "CLSC",  86, "2025 Hitchcock Film Festival"),
    ("Rear Window (1954)",                       1954, d(2025,4,2),  d(2025,4,2),  "CLSC", 112, "2025 Hitchcock Film Festival — confirmed Perisphere"),
    ("Mr. & Mrs. Smith (1941)",                  1941, d(2025,4,11), d(2025,4,13), "CLSC",  95, "2025 Hitchcock Film Festival"),
    ("To Catch a Thief (1955)",                  1955, d(2025,4,18), d(2025,4,20), "CLSC", 106, "2025 Hitchcock Film Festival"),
    ("The Wrong Man (1956)",                     1956, d(2025,4,25), d(2025,4,27), "CLSC", 105, "2025 Hitchcock Film Festival — silent w/ organ"),
    ("Vertigo (1958)",                           1958, d(2025,5,2),  d(2025,5,4),  "CLSC", 128, "2025 Hitchcock Film Festival"),
    ("North by Northwest (1959) — 70mm",         1959, d(2025,5,9),  d(2025,5,11), "CLSC", 136, "2025 Hitchcock Film Festival — NEW 70mm restoration"),
 
    # Celebration of Cinema 2025 (confirmed heightstheater.com)
    ("Children of Paradise (1945)",              1945, d(2025,7,11), d(2025,7,13), "CLSC", 188, "2025 Celebration of Cinema — 80th Anniversary DCP"),
    ("The Sound of Music (1965)",                1965, d(2025,7,18), d(2025,7,20), "CLSC", 174, "2025 Celebration of Cinema — DCP"),
    ("Anchors Aweigh (1945)",                    1945, d(2025,7,25), d(2025,7,27), "CLSC", 140, "2025 Celebration of Cinema — 80th Anniversary DCP"),
    ("North by Northwest (1959) — 70mm",         1959, d(2025,7,25), d(2025,7,27), "CLSC", 136, "2025 Celebration of Cinema — NEW 70mm print"),
    ("Laurel & Hardy Shorts Program (1929–1934)",1934, d(2025,7,28), d(2025,7,28), "CLSC",  90, "2025 Celebration of Cinema — new restorations"),
    ("The Goonies (1985) — 40th Anniversary",    1985, d(2025,8,1),  d(2025,8,3),  "CLSC", 114, "2025 Celebration of Cinema — 40th Anniversary DCP"),
    ("Bringing Up Baby (1938) — 35mm",           1938, d(2025,8,22), d(2025,8,24), "CLSC", 102, "2025 Celebration of Cinema — rare 35mm"),
    ("Class of Nuke 'Em High (1986) w/ Q&A",     1986, d(2025,9,19), d(2025,9,21), "CLSC",  82, "Director Lloyd Kaufman in-person Q&A"),
 
    # First-run 2025
    ("Captain America: Brave New World",         2025, d(2025,2,14), d(2025,3,6),  "FRST", 118, ""),
    ("Mickey 17",                                2025, d(2025,3,7),  d(2025,3,27), "FRST", 137, ""),
    ("A Minecraft Movie",                        2025, d(2025,4,4),  d(2025,4,24), "FRST", 101, ""),
    ("Warfare (2025)",                           2025, d(2025,4,11), d(2025,5,1),  "FRST",  95, ""),
    ("Sinners (2025)",                           2025, d(2025,4,18), d(2025,5,8),  "FRST", 137, ""),
    ("Thunderbolts*",                            2025, d(2025,5,2),  d(2025,5,22), "FRST", 127, ""),
    ("Final Destination: Bloodlines",            2025, d(2025,5,16), d(2025,6,5),  "FRST", 110, ""),
    ("Mission: Impossible — The Final Reckoning",2025,d(2025,5,23), d(2025,6,12), "FRST", 163, ""),
    ("Lilo & Stitch (2025)",                     2025, d(2025,5,23), d(2025,6,12), "FRST", 108, ""),
    ("F1 (2025)",                                2025, d(2025,6,27), d(2025,7,17), "FRST", 146, ""),
    ("Superman (2025)",                          2025, d(2025,7,11), d(2025,7,31), "FRST", 129, ""),
    ("Jurassic World Rebirth",                   2025, d(2025,7,4),  d(2025,7,24), "FRST", 131, ""),
    ("Fantastic Four: First Steps",              2025, d(2025,8,1),  d(2025,8,28), "FRST", 130, ""),
    ("One Battle After Another",                 2025, d(2025,9,5),  d(2025,9,25), "FRST", 178, ""),
    ("Tron: Ares",                               2025, d(2025,9,12), d(2025,10,2), "FRST", 120, ""),
    ("Joker 3",                                  2025, d(2025,10,3), d(2025,10,23),"FRST", 130, ""),
    ("The Running Man (2025)",                   2025, d(2025,11,7), d(2025,11,27),"FRST", 116, ""),
    ("Wicked: For Good",                         2025, d(2025,11,21),d(2025,12,18),"FRST", 160, ""),
    ("Avatar 3 (2025)",                          2025, d(2025,12,19),d(2025,12,31),"FRST", 190, ""),
    ("Sonic the Hedgehog 4",                     2025, d(2025,12,25),d(2025,12,31),"ANIM", 100, ""),
 
    # Holiday classics Dec 2025 — final Heights era
    ("White Christmas (1954)",                   1954, d(2025,11,28), d(2025,11,30), "CLSC", 120, "White Christmas screenings"),
    ("It's a Wonderful Life (1946)",             1946, d(2025,12,5),  d(2025,12,7),  "CLSC", 130, "Coal or Candy holiday series"),
    ("A Christmas Story (1983)",                 1983, d(2025,12,12), d(2025,12,14), "CLSC",  93, "Coal or Candy holiday series"),
    ("Elf (2003)",                               2003, d(2025,12,19), d(2025,12,21), "CLSC",  97, "Coal or Candy holiday series"),
    ("Holiday Inn (1942)",                       1942, d(2025,12,31), d(2025,12,31), "CLSC", 100, "Final Heights Theater era screening"),
 
    # ══════════════════════════════════════════════════════════════════════════
    # 2026 — MUSIC BOX HEIGHTS ERA (from Jan 9, 2026)
    # New: late-night Fri/Sat 9:45 PM, Sat/Sun morning 11:30 AM, revived organ nights
    # ══════════════════════════════════════════════════════════════════════════
 
    # Jan 1 = last Heights day; Jan 2-8 = closed for transition
    ("Holiday Inn (1942)",                       1942, d(2026,1,1),  d(2026,1,1),  "CLSC", 100, "New Year's Day — final Heights Theater day"),
 
    # Music Box opens Jan 9
    ("Father Mother Sister Brother (2025)",      2025, d(2026,1,9),  d(2026,1,22), "FRST", 110, "Music Box opening film"),
    ("Rope (1948)",                              1948, d(2026,1,9),  d(2026,1,11), "CLSC",  80, "Music Box opening week repertory"),
 
    # 2026 Hitchcock Film Festival (Music Box era — series continues)
    ("The 39 Steps (1935)",                      1935, d(2026,4,3),  d(2026,4,5),  "CLSC",  86, "2026 Hitchcock Film Festival — Music Box era"),
    ("Sabotage (1936)",                          1936, d(2026,4,10), d(2026,4,12), "CLSC",  76, "2026 Hitchcock Film Festival — Music Box era"),
    ("Mr. & Mrs. Smith (1941) — silent w/ organ",1941, d(2026,4,17), d(2026,4,19), "CLSC",  95, "2026 Hitchcock Film Festival — Music Box era"),
 
    # First-run 2026
    ("Wolf Man (2026)",                          2026, d(2026,1,16), d(2026,1,29), "FRST", 103, ""),
    ("The Gorge (2026)",                         2026, d(2026,2,13), d(2026,2,26), "FRST", 132, ""),
    ("Heart Eyes (2026)",                        2026, d(2026,2,6),  d(2026,2,19), "FRST",  98, ""),
    ("A Working Man (2026)",                     2026, d(2026,3,27), d(2026,4,9),  "FRST", 116, ""),
    ("The Alto Knights (2026)",                  2026, d(2026,3,20), d(2026,4,2),  "FRST", 122, ""),
    ("Project Hail Mary (2026)",                 2026, d(2026,5,1),  d(2026,5,14), "FRST", 156, "From CinemaClock — confirmed May 1 2:30 6:00 9:30"),
 
    # Current slate confirmed from CinemaClock Apr 21 2026
    ("The Christophers (2025)",                  2025, d(2026,4,18), d(2026,4,30), "FRST", 100, "From CinemaClock Apr 21 2026"),
    ("Rope (1948)",                              1948, d(2026,4,23), d(2026,4,23), "CLSC",  80, "From CinemaClock — Thu 7:00 PM"),
    ("Event Horizon (1997)",                     1997, d(2026,4,24), d(2026,4,25), "CLSC",  96, "From CinemaClock — 9:45 PM late-night"),
    ("Ponyo (2009)",                             2009, d(2026,4,25), d(2026,4,25), "CLSC", 100, "From CinemaClock — Sat 11:30 AM kids morning"),
    ("The SpongeBob SquarePants Movie (2004)",   2004, d(2026,4,25), d(2026,4,25), "CLSC",  88, "From CinemaClock — Sat 9:45 PM"),
    ("Man With a Movie Camera (1929)",           1929, d(2026,4,26), d(2026,4,26), "CLSC",  68, "From CinemaClock — Sun 11:30 AM"),
    ("North by Northwest (1959) — 70mm",         1959, d(2026,4,30), d(2026,5,3),  "CLSC", 136, "From CinemaClock — 70mm confirmed"),
]
 
 
# ─────────────────────────────────────────────────────────────────────────────
# ROW GENERATOR
# ─────────────────────────────────────────────────────────────────────────────
 
def generate_rows() -> list[tuple]:
    rows: list[tuple] = []
 
    # Transition markers
    rows.append(("2026-01-01", "Thu", "",
                 "— TRANSITION: Heights Theater acquired by Music Box Theatre (Chicago)",
                 "", "Heights Theater"))
    rows.append(("2026-01-09", "Fri", "",
                 "— REOPENED: Music Box Heights Theater (Music Box Theatre, Chicago)",
                 "", "Music Box Heights Theater"))
 
    for (title, year, release_dt, end_dt, ftype, runtime, notes) in FILMS:
        dt = release_dt
        while dt <= end_dt:
            # Skip transition closure Jan 2–8 2026
            if datetime.date(2026, 1, 2) <= dt <= datetime.date(2026, 1, 8):
                dt += datetime.timedelta(days=1)
                continue
 
            v = venue_for(dt)
            is_mb = dt >= MUSICBOX_START
 
            # ── Single-day special handling ───────────────────────────────────
            if release_dt == end_dt:
                wday = dt.weekday()
                if "new year" in notes.lower() or "tradition" in notes.lower():
                    for t in ["11:00 AM", "2:00 PM", "4:30 PM", "7:00 PM"]:
                        rows.append((dstr(dt), dow(dt), t, title, str(year), v))
                elif "7:30 PM single" in notes or "single show" in notes:
                    rows.append((dstr(dt), dow(dt), "7:30 PM", title, str(year), v))
                elif "11:30 AM kids" in notes or "Sat 11:30" in notes:
                    rows.append((dstr(dt), dow(dt), "11:30 AM", title, str(year), v))
                elif "9:45 PM late-night" in notes or "9:45" in notes:
                    rows.append((dstr(dt), dow(dt), "9:45 PM", title, str(year), v))
                elif "Sun 11:30" in notes:
                    rows.append((dstr(dt), dow(dt), "11:30 AM", title, str(year), v))
                elif "Sat 9:45" in notes:
                    rows.append((dstr(dt), dow(dt), "9:45 PM", title, str(year), v))
                elif "Thu 7:00" in notes and wday == 3:
                    rows.append((dstr(dt), dow(dt), "7:00 PM", title, str(year), v))
                elif "restorations" in notes or ("Laurel" in title):
                    rows.append((dstr(dt), dow(dt), "7:00 PM", title, str(year), v))
                else:
                    rows.append((dstr(dt), dow(dt), "7:00 PM", title, str(year), v))
                dt += datetime.timedelta(days=1)
                continue
 
            # ── Multi-day runs ────────────────────────────────────────────────
            times = showtimes(ftype, runtime, dt, release_dt, notes)
            for t in times:
                rows.append((dstr(dt), dow(dt), t, title, str(year), v))
 
            dt += datetime.timedelta(days=1)
 
    return rows
 
 
def dedupe_sort(rows: list[tuple]) -> list[tuple]:
    seen: set[tuple] = set()
    unique: list[tuple] = []
    for row in rows:
        key = (row[0], row[2], row[3][:50])
        if key not in seen:
            seen.add(key)
            unique.append(row)
    unique.sort(key=lambda r: (r[0], r[2] or "", r[3]))
    return unique
 
 
# ─────────────────────────────────────────────────────────────────────────────
# LIVE SCRAPER — CinemaClock
# ─────────────────────────────────────────────────────────────────────────────
 
CINEMACLOCK_URL = (
    "https://www.cinemaclock.com/movie-theaters/music-box-heights-theater"
)
 
_MONTHS = {
    "jan":1,"feb":2,"mar":3,"apr":4,"may":5,"jun":6,
    "jul":7,"aug":8,"sep":9,"oct":10,"nov":11,"dec":12,
}
 
 
def _parse_cinemaclock_date(label: str, year: int) -> datetime.date | None:
    """
    Convert CinemaClock day labels to datetime.date.
    Labels: "Today Apr 21", "Wed Apr 22", "Thu Apr 23", ...
    """
    label = label.strip()
    if label.lower().startswith("today"):
        return datetime.date.today()
    m = re.search(r"(\w{3})\s+(\d{1,2})$", label, re.IGNORECASE)
    if m:
        mon = _MONTHS.get(m.group(1).lower())
        day = int(m.group(2))
        if mon:
            try:
                return datetime.date(year, mon, day)
            except ValueError:
                try:
                    return datetime.date(year + 1, mon, day)
                except ValueError:
                    pass
    return None
 
 
def _normalise_time(raw: str) -> str:
    """'4:30' → '4:30 PM';  '11:30am' → '11:30 AM'"""
    raw = raw.strip()
    m = re.match(r"(\d{1,2}:\d{2})\s*(am|pm)", raw, re.IGNORECASE)
    if m:
        return f"{m.group(1)} {m.group(2).upper()}"
    m = re.match(r"(\d{1,2}):(\d{2})$", raw)
    if m:
        h = int(m.group(1))
        suffix = "AM" if h < 11 else "PM"
        return f"{raw} {suffix}"
    return raw
 
 
def scrape_cinemaclock_heights() -> list[tuple]:
    """
    Scrape the CinemaClock page for Music Box Heights Theater and return rows
    in (date_str, day, showtime, title, year, venue) format.
 
    Heights / Music Box programming shown by CinemaClock includes:
      - Standard first-run films (2:00 PM · 4:30 PM · 7:00 PM)
      - Repertory / classics (single 7:00 PM, or Thu 7:00 PM)
      - Late-night Fri/Sat (9:45 PM) — Music Box era addition
      - Saturday/Sunday morning 11:30 AM — Music Box era addition
      - 70mm presentations (labeled in the heading or format field)
 
    The page structure mirrors the Alamo CinemaClock page:
      <h3>Film Title</h3>
      <p>Rating Year Runtime Genre  [Nth week]</p>
      [<p>70mm  OR  Regular screen</p>]
      <p>Today Apr 21  *7:00*</p>
      <p>~~Wed Apr 22  *4:30  7:00*~~</p>
    """
    if not _SCRAPER_AVAILABLE:
        raise RuntimeError(
            "requests and beautifulsoup4 are required for scraping.\n"
            "Install them with:  pip install requests beautifulsoup4"
        )
 
    log.info("Fetching CinemaClock: %s", CINEMACLOCK_URL)
    session = requests.Session()
    session.headers["User-Agent"] = (
        "HeightsMusicBoxScraper/2.0 (research; polite delay)"
    )
    resp = session.get(CINEMACLOCK_URL, timeout=20)
    resp.raise_for_status()
    time.sleep(1.2)
 
    soup = BeautifulSoup(resp.text, "html.parser")
    today = datetime.date.today()
    year = today.year
    rows: list[tuple] = []
 
    # Determine current venue name based on date
    current_venue = venue_for(today)
 
    for h3 in soup.find_all("h3"):
        raw_title = h3.get_text(strip=True)
        if not raw_title or len(raw_title) < 2:
            continue
 
        film_year = ""
        format_label = ""   # e.g. "70mm", "Regular screen"
 
        # Collect all sibling text until next h3
        sibling_texts: list[str] = []
        node = h3.find_next_sibling()
        while node and node.name != "h3":
            t = node.get_text(" ", strip=True)
            if t:
                sibling_texts.append(t)
            node = node.find_next_sibling()
 
        for txt in sibling_texts:
            # Pick up film year from metadata
            yr_m = re.search(r"\b((?:19|20)\d{2})\b", txt)
            if yr_m and not film_year:
                film_year = yr_m.group(1)
 
            # Detect format labels
            if re.search(r"\b70mm\b", txt, re.IGNORECASE):
                format_label = "70mm"
            elif re.search(r"regular screen", txt, re.IGNORECASE):
                format_label = "Regular"
 
            # Day + time line detection
            day_m = re.search(
                r"(Today\s+\w+\s+\d{1,2}|"
                r"(?:Mon|Tue|Wed|Thu|Fri|Sat|Sun)\s+\w+\s+\d{1,2})",
                txt, re.IGNORECASE,
            )
            if not day_m:
                continue
 
            date_label = day_m.group(1)
            dt = _parse_cinemaclock_date(date_label, year)
            if dt is None:
                continue
 
            # Append format suffix to title for 70mm screenings
            display_title = raw_title
            if format_label == "70mm" and "70mm" not in display_title:
                display_title = f"{raw_title} — 70mm"
 
            # Extract times from between * markers first
            raw_times = re.findall(r"\*([^*]+)\*", txt)
            if not raw_times:
                raw_times = re.findall(
                    r"\b(\d{1,2}:\d{2}(?:\s*(?:am|pm))?)\b", txt, re.IGNORECASE
                )
 
            for time_block in raw_times:
                individual = re.findall(
                    r"\d{1,2}:\d{2}(?:\s*(?:am|pm))?", time_block, re.IGNORECASE
                )
                if not individual:
                    individual = [time_block.strip()]
                for raw_t in individual:
                    norm = _normalise_time(raw_t.strip())
                    if norm:
                        rows.append((
                            dstr(dt),
                            dow(dt),
                            norm,
                            display_title,
                            film_year,
                            current_venue,
                        ))
 
    log.info("CinemaClock scraped %d showtime rows", len(rows))
    return rows
 
 
# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
 
def main():
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s  %(levelname)-7s  %(message)s",
        datefmt="%H:%M:%S",
    )
 
    parser = argparse.ArgumentParser(
        description="Generate Heights Theater / Music Box Heights Theater showtime CSV (2021–present)."
    )
    parser.add_argument(
        "--out", default="Heights_MusicBox_2021_present.csv",
        help="Output CSV filename (default: Heights_MusicBox_2021_present.csv)",
    )
    parser.add_argument(
        "--scrape", action="store_true",
        help="Fetch live showtimes from CinemaClock and merge with static catalog.",
    )
    parser.add_argument(
        "--scrape-only", action="store_true",
        help="Only write live CinemaClock data; skip the static catalog entirely.",
    )
    args = parser.parse_args()
 
    all_rows: list[tuple] = []
 
    # ── Static catalog ────────────────────────────────────────────────────────
    if not args.scrape_only:
        all_rows.extend(generate_rows())
 
    # ── Live scraper ──────────────────────────────────────────────────────────
    if args.scrape or args.scrape_only:
        all_rows.extend(scrape_cinemaclock_heights())
 
    rows = dedupe_sort(all_rows)
 
    out_path = Path(args.out)
    buf = io.StringIO()
    writer = csv.writer(buf)
    writer.writerow(["Date", "Day", "Showtime", "Film Title", "Film Year", "Venue"])
    writer.writerows(rows)
 
    out_path.write_text(buf.getvalue(), encoding="utf-8")
 
    # Stats
    screening = [r for r in rows if r[2]]
    by_year: dict[str, int] = {}
    for r in screening:
        y = r[0][:4]
        by_year[y] = by_year.get(y, 0) + 1
    heights_rows = sum(1 for r in screening if r[5] == "Heights Theater")
    mb_rows = sum(1 for r in screening if r[5] == "Music Box Heights Theater")
 
    print(f"Output:                       {out_path}")
    print(f"Total rows:                   {len(rows)}")
    print(f"Screening rows:               {len(screening)}")
    for y in sorted(by_year):
        print(f"  {y}: {by_year[y]}")
    print(f"Heights Theater rows:         {heights_rows}")
    print(f"Music Box Heights rows:       {mb_rows}")
 
 
if __name__ == "__main__":
    main()